# MTRAG Dataset Guide

::::::::::::{div}
:class: proposal-document mtrag-guide-page

MTRAG contains 110 conversations and 842 turn-level tasks across four document collections. These notes cover the files used in the [proposal](Proposal.ipynb), with counts checked against the October 23, 2026 snapshot.

[Data viewer](MTRAG-Viewer.ipynb) · [Inspection script](../data/downloads/inspect_mtrag.py) · [Counts and file hashes](../data/Research/mtrag-inspection-summary.json)

(mtrag-task)=
## A Conversation Becomes Several Tasks

:::::::::{admonition} Task
:class: note

One current user question, its earlier conversation, the supplied passages and a separate reference answer. Each task belongs to a conversation through `conversation_id`.
:::::::::

The first task in `reference.jsonl` asks:

> where do the arizona cardinals play this week

It is labeled `UNANSWERABLE`, has no reference passages, and its reference answer declines to answer. The historical collection cannot establish this week's schedule.

The second task asks whether the Cardinals play outside the US. It is labeled `ANSWERABLE`. Its input includes the first question and response, followed by the new question. [View this conversation](MTRAG-Viewer.ipynb).

```json
{
  "turn": "1",
  "input": [{
    "speaker": "user",
    "text": "where do the arizona cardinals play this week"
  }],
  "contexts": [],
  "Answerability": ["UNANSWERABLE"]
}
```

This excerpt omits the task's metadata and target. The full task ID is `dd6b6ffd177f2b311abe676261279d2f<::>1`.

The history is fixed in the released task. If a candidate model gives a different answer to turn 1, that answer does not become turn 2's history. An experiment using these inputs therefore measures responses to supplied conversations; a live rollout would be a different experiment.

```{figure} ../data/Research/mtrag-task-flow.svg
:alt: Earlier messages, the current question and supplied passages feed a candidate answer. A judge scores the answer, using a reference only when the metric requires it.
:width: 100%

The reference answer is held out during candidate generation.
```

(mtrag-files)=
## Files and Evidence Settings

For generation, start with `mtrag-human/generation_tasks/`. These JSONL files already contain passage text.

| File | Tasks | Passages per task |
| :--- | ---: | :--- |
| `reference.jsonl` | 842 | 0–9 reference passages |
| `RAG.jsonl` | 842 | Five retrieved passages |
| `reference+RAG.jsonl` | 436 | Five passages, including reference evidence |

Reference + RAG is restricted to tasks with at most two reference contexts. Its 436 tasks are a selected subset, so a comparison across all three settings needs matched `task_id` values. See the [generation documentation](https://github.com/IBM/mt-rag-benchmark/blob/53fef6f5917c7e10c436f697e69a955dbc795d81/mtrag-human/generation_tasks/README.md).

Other files used here:

| Path under `mtrag-human/` | Contents |
| :--- | :--- |
| `conversations/conversations.json` | A JSON list of 110 complete conversations |
| `evaluations/reference_subset_with_human_evaluations.json` | A JSON dictionary containing tasks, documents, candidate responses and ratings |
| `retrieval_tasks/` | Collection-specific queries and relevance judgments in JSONL and TSV |

:::::::::{dropdown} Full document collections

The full corpora are needed to rebuild retrieval. IBM recommends `corpora/passage_level/` so passage boundaries match the benchmark.

| Collection | Subject | Documents | Passages |
| :--- | :--- | ---: | ---: |
| ClapNQ | Wikipedia | 4,293 | 183,408 |
| Cloud | Technical documentation | 57,638 | 61,022 |
| FiQA | Finance | 7,661 | 49,607 |
| Govt | Government | 8,578 | 72,422 |

These are [IBM's reported corpus counts](https://github.com/IBM/mt-rag-benchmark/blob/53fef6f5917c7e10c436f697e69a955dbc795d81/corpora/README.md); the full compressed corpora were not recounted here. Retrieval qrels cover answerable and partial tasks.
:::::::::

(mtrag-open)=
## Loading the Files

The [inspection script](../data/downloads/inspect_mtrag.py) downloads five files from the pinned commit and checks their keys and joins. It requires Python 3.9+ with no additional packages.

```bash
python inspect_mtrag.py --cache ./mtrag-cache
```

It writes `inspection-summary.json` and `human-ratings.jsonl` alongside the downloaded files. The summary includes file hashes.

:::::::::{dropdown} Python: load reference tasks

Run this after the inspection script, from the same working directory:

```python
import json
from pathlib import Path
from collections import Counter

cache = Path("mtrag-cache")
path = cache / "mtrag-human__generation_tasks__reference.jsonl"
with path.open(encoding="utf-8") as stream:
    tasks = [json.loads(line) for line in stream if line.strip()]

by_id = {task["task_id"]: task for task in tasks}
assert len(by_id) == len(tasks) == 842
assert len({t["conversation_id"] for t in tasks}) == 110
assert len({(t["conversation_id"], int(t["turn"]))
            for t in tasks}) == len(tasks)

for task in tasks:
    assert task["input"][-1]["speaker"] == "user"
    assert len(task["input"]) == 2 * int(task["turn"]) - 1

print(Counter(t["Collection"] for t in tasks))
print(Counter(label for t in tasks for label in t["Answerability"]))
```
:::::::::

### Fields That Need Attention

The actual keys are `Collection` and `Answerability`, although the README's example uses lowercase names. `turn` is stored as a string; convert it to an integer before sorting.

| Field | Contents |
| :--- | :--- |
| `task_id` | Unique identifier within an evidence setting |
| `conversation_id` | Shared identifier for turns in the same conversation |
| `turn` | Current turn number, stored as text |
| `input[:-1]` | Earlier user and agent messages |
| `input[-1]` | Current user question |
| `contexts` | Supplied passages, including text and document IDs |
| `targets` | Released reference answer(s) |
| `Answerability` | Answerable, partial, unanswerable or conversational |
| `Question Type`, `Multi-Turn` | Question and conversation labels |
| `Collection` | Source collection |

Reference metadata can still name Mixtral: creation involved model generation and human review. A reference answer is comparison text, not a human quality score.

(mtrag-distributions)=
## Counts from the Reference File

All 842 task IDs and conversation–turn pairs are unique. Conversations contain 3–12 tasks, averaging 7.65. There are 110 first turns and 732 later turns.

::::::::::{grid} 1 1 2 2

:::::::::{grid-item-card} Collection

| Collection | Tasks |
| :--- | ---: |
| ClapNQ | 224 |
| Cloud | 205 |
| FiQA | 199 |
| Govt | 214 |
:::::::::

:::::::::{grid-item-card} Answerability

| Label | Tasks |
| :--- | ---: |
| ANSWERABLE | 709 |
| PARTIAL | 68 |
| UNANSWERABLE | 55 |
| CONVERSATIONAL | 10 |
:::::::::
::::::::::

The 65 empty reference-context lists correspond exactly to the 55 unanswerable and 10 conversational tasks. They are expected cases, not rows to discard as missing data. Full RAG supplies five passages even for these tasks; retrieved text alone does not make a question answerable.

The multi-turn labels are 622 Follow-up, 110 Clarification and 110 N/A. Every task in this release has `2 × turn − 1` input messages and ends with a user question.

:::::::::{dropdown} Python: count passages

```python
passage_counts = Counter(len(t["contexts"]) for t in tasks)
print(dict(sorted(passage_counts.items())))
# {0: 65, 1: 105, 2: 266, 3: 243, 4: 96,
#  5: 36, 6: 21, 7: 8, 8: 1, 9: 1}

empty = [t for t in tasks if not t["contexts"]]
print(Counter(a for t in empty for a in t["Answerability"]))
# {'UNANSWERABLE': 55, 'CONVERSATIONAL': 10}
```
:::::::::

(mtrag-human)=
## Human Evaluations

The human evaluation file contains 159 tasks from 20 conversations and 477 task–model records: one each for the reference, Llama 3.1 405B and GPT-4o responses. It uses the reference evidence setting.

`tasks` contains passage IDs; the text is in `documents`. Each record in `evaluations` has a `task_id`, `model_id`, `model_response` and `annotations`. Join to the generation file on `task_id` to recover `conversation_id` and turn number.

| Human metric | Scale | Individual ratings |
| :--- | :--- | ---: |
| Naturalness | Ordered, 1–4 | 1,413 |
| Appropriateness | Ordered, 1–4 | 1,413 |
| Completeness | Ordered, 1–4 | 1,411 |
| Faithfulness | Ordered, 1–4 | 1,419 |
| Win rate | 0–100 | 1,431 |

Three ratings per response would give 1,431 entries per metric. Several metrics have fewer; missing annotations need their own denominator. The four ordered scales run from “No” to “Yes,” with median aggregation specified in the released metadata.

:::::::::{dropdown} Python: join a response to its task and passages

Continue after the earlier loading example:

```python
human_path = cache / (
    "mtrag-human__evaluations__"
    "reference_subset_with_human_evaluations.json"
)
with human_path.open(encoding="utf-8") as stream:
    human = json.load(stream)

documents = {d["document_id"]: d for d in human["documents"]}
subset = {t["task_id"]: t for t in human["tasks"]}
evaluation = human["evaluations"][0]
source_task = by_id[evaluation["task_id"]]
subset_task = subset[evaluation["task_id"]]

print(source_task["conversation_id"], int(source_task["turn"]))
print(evaluation["model_id"])
print(evaluation["model_response"])
print(evaluation["annotations"]["faithfulness"])
for context in subset_task["contexts"]:
    print(documents[context["document_id"]]["text"])
```
:::::::::

The file also includes automatic metrics. Select human metric names from `metrics` where `author == "human"`. Each human metric's annotations are keyed by annotator ID. The inspection script retains these as one row per task, model, metric and annotator.

(mtrag-research)=
## Implications for the Proposal

For reliability estimation, the candidate answers need to be held fixed and scored across the chosen judges, prompts and repeated calls. The public evaluation table does not provide all of those experimental combinations. A self-preference analysis also needs both matching and non-matching judge–agent families.

For metric redundancy, split by **conversation**, keeping its turns together. Otherwise shared history can leak into both training and evaluation folds.

For human-label planning, the available subset supplies a pilot based on 20 conversation clusters. Pair human and judge scores on the same candidate answer and metric, align their scales, and check whether this subset represents the intended population before extrapolating label counts.

(mtrag-sources)=
## Sources

This inspection uses the original `mtrag-human/` release. MTRAG-UN is a separate extension and is not included in these counts.

- [MTRAG paper](https://aclanthology.org/2025.tacl-1.36/)
- [Human data documentation](https://github.com/IBM/mt-rag-benchmark/blob/53fef6f5917c7e10c436f697e69a955dbc795d81/mtrag-human/README.md)
- [Generation files](https://github.com/IBM/mt-rag-benchmark/tree/53fef6f5917c7e10c436f697e69a955dbc795d81/mtrag-human/generation_tasks)
- [Human evaluation file](https://github.com/IBM/mt-rag-benchmark/blob/53fef6f5917c7e10c436f697e69a955dbc795d81/mtrag-human/evaluations/reference_subset_with_human_evaluations.json)

Commit: `53fef6f5917c7e10c436f697e69a955dbc795d81`. Inspected October 23, 2026. The [saved summary](../data/Research/mtrag-inspection-summary.json) records counts and file hashes; the script checks task identity, history length, setting membership and joins between tasks, passages and evaluations.

::::::::::::
